# Phase 1: Environment Setup, Kaggle Data Pipeline, and EDA

This notebook covers the Day 1 deliverables for the **Multimodal Medical Symptom Triage** project:
1. **Environment Setup**: Install and import required libraries for T4 GPU execution.
2. **Data Pipeline**: Download datasets for Layer 1 (Images), Layer 2 (Symptom Text), and Layer 3 (Severity) using the Kaggle API.
3. **Exploratory Data Analysis (EDA)**: Class distributions, dimensions, and balance analysis.
4. **Image Preprocessing**: Pre-processing and data augmentations (224x224 resize, normalization, transforms).

### Step 1: Install Required Libraries

In [ ]:
# Install core packages
!pip install -q torch torchvision transformers datasets scikit-learn pandas numpy matplotlib seaborn kaggle

### Step 2: Configure Kaggle Credentials

To download datasets directly, you need a Kaggle API Token. 
1. Go to Kaggle -> Your Account -> Create New API Token (downloads `kaggle.json`).
2. Run the cell below, click "Choose Files", and upload `kaggle.json`.

In [ ]:
from google.colab import files
import os

# Upload kaggle.json
if not os.path.exists('/content/kaggle.json'):
    uploaded = files.upload()

# Set credentials path
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured successfully.")

### Step 3: Download and Extract Datasets

In [ ]:
import zipfile

# Create directories
os.makedirs('/content/datasets/ham10000', exist_ok=True)
os.makedirs('/content/datasets/symptom2disease', exist_ok=True)
os.makedirs('/content/datasets/severity', exist_ok=True)
os.makedirs('/content/datasets/symptom_binary', exist_ok=True)

print("Downloading datasets...")
# Layer 1: Image (HAM10000)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/datasets/ham10000

# Layer 2: Text (Symptom2Disease & Disease Prediction)
!kaggle datasets download -d niyarrbarman/symptom2disease -p /content/datasets/symptom2disease
!kaggle datasets download -d kaushil268/disease-prediction-using-machine-learning -p /content/datasets/symptom_binary

# Layer 3: Severity (Disease Diagnosis & Severity)
!kaggle datasets download -d s3programmer/disease-diagnosis-dataset -p /content/datasets/severity

print("Extracting archives...")
def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extracted {os.path.basename(zip_path)} to {extract_to}")

extract_zip('/content/datasets/ham10000/skin-cancer-mnist-ham10000.zip', '/content/datasets/ham10000')
extract_zip('/content/datasets/symptom2disease/symptom2disease.zip', '/content/datasets/symptom2disease')
extract_zip('/content/datasets/symptom_binary/disease-prediction-using-machine-learning.zip', '/content/datasets/symptom_binary')
extract_zip('/content/datasets/severity/disease-diagnosis-dataset.zip', '/content/datasets/severity')

print("All datasets downloaded and extracted.")

### Step 4: Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# 1. Layer 1 - Skin Cancer (HAM10000) Metadata EDA
ham_metadata = pd.read_csv('/content/datasets/ham10000/HAM10000_metadata.csv')
print("=== HAM10000 Metadata ===")
print(ham_metadata.info())
print("\nClass distribution:")
print(ham_metadata['dx'].value_counts())

# Plot HAM10000 Class Balances
plt.figure(figsize=(10, 5))
sns.countplot(data=ham_metadata, x='dx', order=ham_metadata['dx'].value_counts().index, palette='viridis')
plt.title('HAM10000 Lesion Class Distribution (Highly Imbalanced)')
plt.ylabel('Count')
plt.xlabel('Diagnosis Class')
plt.show()

# 2. Layer 2 - Symptom2Disease Text EDA
s2d_df = pd.read_csv('/content/datasets/symptom2disease/Symptom2Disease.csv')
# Standardize column names
s2d_df = s2d_df.drop(columns=['Unnamed: 0'], errors='ignore')
print("\n=== Symptom2Disease Dataset ===")
print(s2d_df.head())
print(f"\nTotal records: {len(s2d_df)}")
print(f"Unique diseases: {s2d_df['label'].nunique()}")
print(s2d_df['label'].value_counts())

# 3. Layer 3 - Disease Diagnosis + Severity EDA
severity_df = pd.read_csv('/content/datasets/severity/Disease_Diagnosis_and_Severity.csv')
print("\n=== Disease Severity Dataset ===")
print(severity_df.info())
print(severity_df.head())
print("\nSeverity labels distribution:")
print(severity_df['Severity'].value_counts())

# Plot Severity Labels
plt.figure(figsize=(6, 4))
sns.countplot(data=severity_df, x='Severity', palette='rocket')
plt.title('Severity Classes Distribution')
plt.show()

### Step 5: Build Preprocessing Pipeline

Here, we create the PyTorch Dataset for HAM10000, organizing images and applying data augmentations to balance out the training representation.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# Map image IDs to their paths
image_id_path_map = {}
image_paths = glob.glob('/content/datasets/ham10000/HAM10000_images_part_1/*.jpg') + \
              glob.glob('/content/datasets/ham10000/HAM10000_images_part_2/*.jpg')

for path in image_paths:
    img_id = os.path.splitext(os.path.basename(path))[0]
    image_id_path_map[img_id] = path

ham_metadata['path'] = ham_metadata['image_id'].map(image_id_path_map)
# Drop missing path files if any
ham_metadata = ham_metadata.dropna(subset=['path']).reset_index(drop=True)

# Label Map for classes
dx_map = {label: idx for idx, label in enumerate(ham_metadata['dx'].unique())}
ham_metadata['label'] = ham_metadata['dx'].map(dx_map)
print("Class Map:", dx_map)

# Train/Val Split
train_df, val_df = train_test_split(ham_metadata, test_size=0.2, random_state=42, stratify=ham_metadata['label'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

# Transforms with Augmentations (to handle imbalance and variations)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['label']
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# Create Datasets and Dataloaders
train_dataset = HAM10000Dataset(train_df, transform=train_transforms)
val_dataset = HAM10000Dataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Dataloaders initialized successfully.")